# 🧹 02. Data Cleaning & Normalization
### Amazon ML Challenge 2026: Multi-Source Business Entity Resolution

In this notebook, we demonstrate the deterministic text normalization pipeline:
1. **Unicode NFKD normalization** (stripping combining diacritics / accents, e.g. `é` -> `e`).
2. **Legal Corporate Suffix Normalization & Stripping** (`Inc.`, `LLC`, `Corp`, `Limited`, `GmbH`, etc.).
3. **Address standardizations & digit extraction**.
4. **Open-set country normalization** (supporting US, IN, FR).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "amazon" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.preprocessing.legal_terms import strip_legal_suffixes

from src.preprocessing.address_normalizer import extract_address_digits, normalize_address
from src.preprocessing.country_normalizer import normalize_country
from src.preprocessing.text_normalizer import normalize_business_name, strip_accents

## 1. Testing Unicode & Diacritic Normalization
Accents and combining characters must be cleanly stripped across international entities.

In [ ]:
messy_names = [
    "Café de Paris S.A.R.L.",
    "Münchener Rückversicherungs-Gesellschaft AG",
    "Société Générale (France) S.A.",
    "Wal-Mart Stores, Inc. #482",
    "TATA CONSULTANCY SERVICES LIMITED",
    "St. John's Health Center & Clinic, LLC"
]

results = []
for name in messy_names:
    norm_name = normalize_business_name(name)
    clean_name = strip_legal_suffixes(norm_name)
    results.append({
        "Original": name,
        "Normalized": norm_name,
        "Legal Stripped": clean_name
    })

df_clean = pd.DataFrame(results)
display(df_clean)

## 2. Address Normalization & Digit Extraction
Street types and abbreviations are normalized, and house/suite numbers are extracted.

In [ ]:
sample_addresses = [
    "123 North Main Street, Apt. 4B, New York, NY 10001",
    "P.O. Box 789, Suite 100, San Francisco, CA",
    "Plot No. 42, Sector 18, Gurugram, Haryana 122015",
    "14 Boulevard Saint-Germain, 75005 Paris, France"
]

addr_results = []
for addr in sample_addresses:
    norm_addr = normalize_address(addr)
    digits = extract_address_digits(addr)
    addr_results.append({
        "Original Address": addr,
        "Normalized Address": norm_addr,
        "Extracted Digits": digits
    })

display(pd.DataFrame(addr_results))

## 3. Verify Cleaned Interim Parquet Files
Check the compressed Parquet datasets prepared in `data/interim/`.

In [ ]:
from src.utils.io import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")
interim_dir = PROJECT_ROOT / cfg["paths"]["interim_dir"]

for p in sorted(interim_dir.glob("*.parquet")):
    df = pd.read_parquet(p)
    print(f"File: {p.name:<28} Shape: {str(df.shape):<18} Size: {p.stat().st_size / (1024*1024):.1f} MB")